In [ ]:

import re
from dataclasses import dataclass
from typing import Dict, List, Set, Tuple, Optional
import pandas as pd

# ----------------------------
# 1) Test DB (toy examples)
# ----------------------------
# TEST_DB: Dict[str, str] = {
#     # Based on your examples (slightly expanded)
#     "N01i":  "(SHORT ASC FAST) (PEAK FLAT PEAK)",
#     "N01ii": "(SHORT ASC FAST) (GAP) (PEAK FLAT)",
#     "N01iii":"(SHORT FLAT) (PEAK FLAT)",
#     "N02":   "(BB SBI_TIGHT) (SQUIGGLE) (SHORT ASC FAST)",
#     "N03":   "(DESC SBI_DEC)",
#     "N04":   "(PEAK LEFT LARGE FLAT) (SHORT ASC SBI_TIGHT)",
#     "N05i":  "(UPSWEEP FLAT) (UPSWEEP FLAT ASC) (SHORT SBI_TIGHT ASC)",
#     "N05ii": "(FLAT) (ASC SLOW) (SHORT PEAK)",
#     "N07ii": "(UPSWEEP FLAT) (ASC SBI_INC)",
#     "N07iv": "(UPSWEEP FLAT) (ASC SBI_WIDE)",
#     "N16ii": "(ASC) (FLAT ASC) (SHORT SBI_TIGHT) (ASC SBI_WIDE) (PEAK RIGHT LARGE SBI_WIDE) (SHORT SBI_TIGHT)",
#     "N23ii": "(FLAT SBI_WIDE) (DOWNSWEEP FLAT DOWNSWEEP)",
#     "N25":   "(SBI_TIGHT ASC) (DESC SBI_WIDE BIPHO) (SHORT DIP) (SBI_TIGHT FLAT) (PEAK)",
# }

In [ ]:
pd.set_option('display.max_columns', None)
parts_v4 = pd.read_csv("./data/parts_manual_labels_v4.csv")
parts_v4 = parts_v4[parts_v4["checked_v4"] == True]

DB = dict(zip(parts_v4['Call'], parts_v4['tokens_string']))


In [33]:
DB

{'N01i': '(BB) (SHORT ASC FAST | FLAT) (PEAK FLAT PEAK | PEAK FLAT) ',
 'N01ii': '(BB SHORT) (SHORT ASC FAST | GAP) (PEAK FLAT) ',
 'N01iii': '(BB) (SHORT FLAT ?) (PEAK FLAT) ',
 'N02': '(BB  SBI_TIGHT) (SQUIGGLE) (SHORT ASC FAST) ',
 'N03': '(BB SHORT) (GAP SHORT) (DESC SBI_DEC) ',
 'N04': '(LEFT PEAK LARGE , FLAT) (SHORT ASC SBI_TIGHT) ',
 'N05i': '(UPSWEEP FLAT | UPSWEEP FLAT , ASC) (SHORT SBI_TIGHT , ASC) ',
 'N05ii': '(FLAT | ASC SLOW) (BB SHORT) (SHORT PEAK) (BB SHORT) ',
 'N07i': '(BB) (UPSWEEP , FLAT) ',
 'N07ii': '(BB) (UPSWEEP FLAT) (ASC SBI_INC) ',
 'N07iii': '(BB) (UPSWEEP FLAT) (ASC SBI_INC) ',
 'N07iv': '(BB) (UPSWEEP FLAT) (ASC SBI_WIDE) ',
 'N08i': '(PULSED) (LEFT PEAK) ',
 'N08ii': '(PULSED) (SHORT FLAT SBI_TIGHT) ',
 'N08iii': '(PULSED) (PEAK , FLAT | DESC) ',
 'N08iv': '(PULSED) (SHORT SMALL PEAK) (BB) ',
 'N09i': '(BB) (SHORT ASC FAST) (FLAT | ASC SLOW) (SHORT ASC FAST) ',
 'N10': '(BB) (SHORT ?) (LARGE PEAK , FLAT) (BB SHORT ?) ',
 'N12': '(BB SHORT) (FLAT SBI_TIGH

In [20]:
DB

{'N01i': '(BB) (SHORT ASC FAST | FLAT) (PEAK FLAT PEAK | PEAK FLAT) ',
 'N01ii': '(BB SHORT) (SHORT ASC FAST | GAP) (PEAK FLAT) ',
 'N01iii': '(BB) (SHORT FLAT ?) (PEAK FLAT) ',
 'N02': '(BB  SBI_TIGHT) (SQUIGGLE) (SHORT ASC FAST) ',
 'N03': '(BB SHORT) (GAP SHORT) (DESC SBI_DEC) ',
 'N04': '(LEFT PEAK LARGE , FLAT) (SHORT ASC SBI_TIGHT) ',
 'N05i': '(UPSWEEP FLAT | UPSWEEP FLAT , ASC) (SHORT SBI_TIGHT , ASC) ',
 'N05ii': '(FLAT | ASC SLOW) (BB SHORT) (SHORT PEAK) (BB SHORT) ',
 'N07i': '(BB) (UPSWEEP , FLAT) ',
 'N07ii': '(BB) (UPSWEEP FLAT) (ASC SBI_INC) ',
 'N07iii': '(BB) (UPSWEEP FLAT) (ASC SBI_INC) ',
 'N07iv': '(BB) (UPSWEEP FLAT) (ASC SBI_WIDE) ',
 'N08i': '(PULSED) (LEFT PEAK) ',
 'N08ii': '(PULSED) (SHORT FLAT SBI_TIGHT) ',
 'N08iii': '(PULSED) (PEAK , FLAT | DESC) ',
 'N08iv': '(PULSED) (SHORT SMALL PEAK) (BB) ',
 'N09i': '(BB) (SHORT ASC FAST) (FLAT | ASC SLOW) (SHORT ASC FAST) ',
 'N10': '(BB) (SHORT ?) (LARGE PEAK , FLAT) (BB SHORT ?) ',
 'N12': '(BB SHORT) (FLAT SBI_TIGH

In [ ]:



# ---- run demo ----
index = build_index(DB)

# queries = [
#     "ASC FAST PEAK FLAT",                 # should match N01i/N01ii-ish
#     "UPSWEEP FLAT ASC SBI_INC",            # should match N07ii/N07iii-like
#     "BB SBI_TIGHT SQUIGGLE",               # should match N02
#     "DESC SBI_WIDE +BIPHO -GAP",           # should match N25 style; requires BIPHO, forbids GAP
#     "DOWNSWEEP FLAT",                      # should match N23ii
#     "ASC SBI_WIDE PEAK RIGHT LARGE",       # should match N16ii-ish
# ]
queries = [
    "SQUIGGLE"    # should match N16ii-ish
]

for q in queries:
    print("\nQUERY:", q)
    display(search_calls(q, index, topk=8))



QUERY: SQUIGGLE
{'SQUIGGLE'} set() set()


,Call,Score,Repr,Matched,Missing(optional downweighted),Missing(core)
0,N02,0.264317,(BB SBI_TIGHT) (SQUIGGLE) (SHORT ASC FAST),SQUIGGLE,,
1,N13,0.255319,(BB) (SHORT ASC) (SQUIGGLE | PEAK PEAK) (BB SH...,SQUIGGLE,,
